# Барановський Богдан ФБ-44 Лабораторна робота №2 - Наука про дані: підготовчий етап

## **Частина 2.** Завдання: <br>
* Завантажити та відкрити (вручну або через запропонований скрипт на сайті) наступний датасет: Individual Household Electric Power Consumption Dataset 
* Здійснити data cleaning
* Окремими функціями сформувати вибірки:
    * Обрати всі записи, у яких загальна активна споживана потужність перевищує 5 кВт.
    * Обрати всі записи, у яких сила струму лежить в межах 19-20 А, для них виявити ті, у яких пральна машина та холодильних споживають більше, ніж бойлер та кондиціонер.
    * Обрати випадковим чином 500000 записів (без повторів елементів вибірки), для них обчислити середні величини усіх 3-х груп споживання електричної енергії
    * Обрати ті записи, які після 18-00 споживають понад 6 кВт за хвилину в середньому, серед відібраних визначити ті, у яких основне споживання електроенергії у вказаний проміжок часу припадає на пральну машину, сушарку, холодильник та освітлення (група 2 є найбільшою), а потім обрати кожен третій результат із першої половини та кожен четвертий результат із другої половини.
* Пронормувати та стандартизувати вибраний датасет
* Підрахувати коефіцієнт Пірсона та Спірмена для двох integer/real атрибутів.
* Провести One Hot Encoding категоріального атрибута.


Завантажили файл household_power_consumption.txt та здійснюємо data cleaning:

In [1]:
import pandas as pd
import pandas as pd
import numpy as np
import timeit

def load_and_clean():
    df = pd.read_csv('household_power_consumption.txt', sep=';', low_memory=False)

    df.replace('?', np.nan, inplace=True)
    df = df.dropna()

    cols = ['Global_active_power', 'Global_reactive_power', 'Voltage', 'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']
    for col in cols:
        df[col] = df[col].astype(float)

    return df

start_t = timeit.default_timer()
df = load_and_clean()
print(f"Exec time - {timeit.default_timer() - start_t:.4f} secs")
display(df.head())

Exec time - 5.6781 secs


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,16/12/2006,17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
4,16/12/2006,17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0


Обрати всі записи, у яких загальна активна споживана потужність перевищує 5 кВт.

In [2]:
def filter_power(data):
    return data[data['Global_active_power'] > 5.0]

start_t = timeit.default_timer()
res1 = filter_power(df)
print(f"Exec time - {timeit.default_timer() - start_t:.4f} secs")
display(res1.head())

Exec time - 0.0070 secs


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
11,16/12/2006,17:35:00,5.412,0.470,232.78,23.2,0.0,1.0,17.0
12,16/12/2006,17:36:00,5.224,0.478,232.99,22.4,0.0,1.0,16.0


Обрати всі записи, у яких сила струму лежить в межах 19-20 А, для них виявити ті, у яких пральна машина та холодильних споживають більше, ніж бойлер та кондиціонер.

In [3]:
def filter_intensivity(data):
    res = data[(data['Global_intensity'] >= 19) & (data['Global_intensity'] <= 20)]
    res = res[res['Sub_metering_2'] > res['Sub_metering_3']]
    return res

start_t = timeit.default_timer()
res2 = filter_intensivity(df)
print(f"Exec time -  {timeit.default_timer() - start_t:.4f} secs")
display(res2.head())

Exec time -  0.0096 secs


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
45,16/12/2006,18:09:00,4.464,0.136,234.66,19.0,0.0,37.0,16.0
460,17/12/2006,01:04:00,4.582,0.258,238.08,19.6,0.0,13.0,0.0
464,17/12/2006,01:08:00,4.618,0.104,239.61,19.6,0.0,27.0,0.0
475,17/12/2006,01:19:00,4.636,0.140,237.37,19.4,0.0,36.0,0.0
476,17/12/2006,01:20:00,4.634,0.152,237.17,19.4,0.0,35.0,0.0


Обрати випадковим чином 500000 записів (без повторів елементів вибірки), для них обчислити середні величини усіх 3-х груп споживання електричної енергії

In [4]:
def random_sample_avg(data):
    sample = data.sample(n=500000, replace=False)
    return sample['Sub_metering_1'].mean(), sample['Sub_metering_2'].mean(), sample['Sub_metering_3'].mean()

start_t = timeit.default_timer()
a1, a2, a3 = random_sample_avg(df)
print(f"Exec time - {timeit.default_timer() - start_t:.4f} secs")
print(f"Sub_metering_1 avg: {a1:.2f}")
print(f"Sub_metering_2 avg: {a2:.2f}")
print(f"Sub_metering_3 avg: {a3:.2f}")

Exec time - 0.1954 secs
Sub_metering_1 avg: 1.12
Sub_metering_2 avg: 1.29
Sub_metering_3 avg: 6.46


Обрати ті записи, які після 18-00 споживають понад 6 кВт за хвилину в середньому, серед відібраних визначити ті, у яких основне споживання електроенергії у вказаний проміжок часу припадає на групу 2, а потім обрати кожен третій результат із першої половини та кожен четвертий результат із другої половини.

In [5]:
def filter_evening(data):
    res = data[(data['Time'] > '18:00:00') & (data['Global_active_power'] > 6.0)]
    res = res[(res['Sub_metering_2'] > res['Sub_metering_1']) & (res['Sub_metering_2'] > res['Sub_metering_3'])]
    res = res.reset_index(drop=True)

    half = len(res) // 2
    part1 = res.iloc[:half:3]
    part2 = res.iloc[half::4]

    return pd.concat([part1, part2])

start_t = timeit.default_timer()
res3 = filter_evening(df)
print(f"Exec time: {timeit.default_timer() - start_t:.4f} secs")
display(res3.head())

Exec time: 0.1144 secs


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,16/12/2006,18:05:00,6.052,0.192,232.93,26.2,0.0,37.0,17.0
3,16/12/2006,18:08:00,6.308,0.116,232.25,27.0,0.0,36.0,17.0
6,28/12/2006,20:58:00,6.386,0.374,236.63,27.0,1.0,36.0,17.0
9,28/12/2006,21:02:00,8.088,0.262,235.50,34.4,1.0,72.0,17.0
12,28/12/2006,21:05:00,7.230,0.152,235.22,30.6,1.0,73.0,17.0


Пронормувати та стандартизувати вибраний датасет

In [6]:
def norm_stand(data, col):
    res = data[[col]].copy()

    min_v = res[col].min()
    max_v = res[col].max()
    res['Normalized'] = (res[col] - min_v) / (max_v - min_v)

    mean_v = res[col].mean()
    std_v = res[col].std()
    res['Standardized'] = (res[col] - mean_v) / std_v

    return res

start_t = timeit.default_timer()
res4 = norm_stand(df, 'Global_active_power')
print(f"Exec time - {timeit.default_timer() - start_t:.4f} secs")
display(res4.head())

Exec time - 0.0641 secs


,Global_active_power,Normalized,Standardized
0,4.216,0.374796,2.955076
1,5.360,0.478363,4.037084
2,5.374,0.479631,4.050325
3,5.388,0.480898,4.063566
4,3.666,0.325005,2.434881


Підрахувати коефіцієнт Пірсона та Спірмена для двох integer/real атрибутів.

In [9]:
def calc_corr(data, col1, col2):
    pearson = data[col1].corr(data[col2], method='pearson')
    spearman = data[col1].corr(data[col2], method='spearman')
    return pearson, spearman

start_t = timeit.default_timer()
p, s = calc_corr(df, 'Global_active_power', 'Global_intensity')
print(f"Exec time - {timeit.default_timer() - start_t:.4f} secs")
print(f"Pearson: {p:.4f}")
print(f"Spearman: {s:.4f}")

Exec time - 1.2663 secs
Pearson: 0.9989
Spearman: 0.9954


Провести One Hot Encoding категоріального атрибута.

In [8]:
def perform_ohe(data):
    res = data[['Date', 'Time', 'Global_active_power']].copy()
    res['Hour'] = res['Time'].str.split(':').str[0]
    return pd.get_dummies(res, columns=['Hour'], dtype=int)

start_t = timeit.default_timer()
res5 = perform_ohe(df)
print(f"Exec time - {timeit.default_timer() - start_t:.4f} secs")
display(res5.head())

Exec time - 2.7631 secs


,Date,Time,Global_active_power,Hour_00,Hour_01,Hour_02,Hour_03,Hour_04,Hour_05,Hour_06,...,Hour_14,Hour_15,Hour_16,Hour_17,Hour_18,Hour_19,Hour_20,Hour_21,Hour_22,Hour_23
0,16/12/2006,17:24:00,4.216,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
1,16/12/2006,17:25:00,5.360,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
2,16/12/2006,17:26:00,5.374,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
3,16/12/2006,17:27:00,5.388,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
4,16/12/2006,17:28:00,3.666,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0


Отже, все працює коректно